In [0]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.shuffle.partitions", "40")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")


start_date = "2025-01-01"
end_date   = "2025-12-31"
recency_anchor = F.to_date(F.lit("2026-01-01"))
snapshot_date_id = F.date_format(recency_anchor, "yyyyMMdd").cast("int")
start_2024 = "2024-01-01"
end_2024   = "2024-12-31"
start_2023 = "2023-01-01"
end_2023   = "2023-12-31"

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_sessions = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_films = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_film")
T_ticket = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_ticket_type")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_genre = spark.table("cpx_dataanalytics_gold.customer360.film_composite_genres")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")
T_emails = spark.table("cpx_dataanalytics_gold.marketing.cpx_d_customer_email_address")
T_rfm = spark.table("cpx_dataanalytics_gold.customer360.c360_f_rfm_model")
T_customer = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_customer")

In [0]:
tmp1 = (
    T_trans.alias("t")
    .join(
        T_sessions.alias("s"),
        F.col("t.SessionSkey") == F.col("s.sessionskey"),
        "left"
    )
    .join(
        T_locations.alias("l"),
        F.col("t.LocationId") == F.col("l.LocationID"),
        "left"
    )
    .join(
        T_date.alias("d"),
        F.col("t.VisitDateId") == F.col("d.dateid"),
        "left"
    )
    .filter(F.col("t.CDE_ID").isNotNull())
    .withColumn("CDE_Visit_ID_MNL", F.concat_ws("|",
        F.col("t.CDE_ID").cast("string"),
        F.col("t.VisitDateId").cast("string"),
        F.col("t.LocationId").cast("string")
        )
    )
    # safer: ensure datefull is treated as a DATE before between()
    .filter(F.to_date(F.col("d.datefull")).between(F.lit(start_date), F.lit(end_date)))
    .filter(F.col("l.LC_Location_Type_Description") == F.lit("Cineplex Theatre"))
    .filter((F.col("t.SCENEFLAG") == "Yes") | (F.col("t.CineClubFLAG") == "Yes"))
    .groupBy(F.col("t.CDE_ID"))
    .agg(
        F.countDistinct(
            F.when(F.col("t.Net_Revenue") > 0, F.col("CDE_Visit_ID_MNL"))
        ).alias("No_of_Visits"),

        F.countDistinct(
            F.when(
                (F.col("t.Net_Revenue") > 0) & (F.col("t.ConcessionFLAG") == "Yes"),
                F.col("CDE_Visit_ID_MNL")
            )
        ).alias("Visits_with_Concession"),

        F.sum(F.col("t.Net_Revenue")).alias("Net_Revenue"),

        F.sum(
            F.when(
                (F.col("s.Auditorium_Experience_ID").isin([1, 5, 6])) &
                (F.col("t.TicketFLAG") == "Yes"),
                F.col("t.Quantity")
            ).otherwise(F.lit(0))
        ).alias("No_of_Regular_Tickets"),

        F.sum(
            F.when(
                (F.col("s.Auditorium_Experience_ID").isin([2, 3, 4, 7, 8])) &
                (F.col("t.TicketFLAG") == "Yes"),
                F.col("t.Quantity")
            ).otherwise(F.lit(0))
        ).alias("No_of_Premium_Tickets"),

        F.sum(
            F.when(F.col("t.ConcessionFLAG") == "Yes", F.col("t.Net_Revenue"))
             .otherwise(F.lit(0))
        ).alias("Net_Revenue_Concession"),

        F.sum(
            F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Net_Revenue"))
             .otherwise(F.lit(0))
        ).alias("Net_Revenue_Tickets"),

        F.max(F.to_date(F.col("d.datefull"))).alias("Last_Visit_Date")
    )
    # HAVING SUM(Net_Revenue) > 0
    .filter(F.col("Net_Revenue") > 0)
    .withColumn(
        "Revenue_per_Visit",
        F.when(F.col("No_of_Visits") > 0, F.col("Net_Revenue") / F.col("No_of_Visits"))
         .otherwise(F.lit(0))
    )
    .withColumn("Recency", F.datediff(recency_anchor, F.col("Last_Visit_Date")))
    .drop("Last_Visit_Date")
)


# ---------------------------------------------------------
# 2) tmp: R/F/M scores
# ---------------------------------------------------------
tmp = (
    tmp1
    .withColumn(
        "R_score",
        F.when(F.col("Recency") >= 180, 1)
         .when(F.col("Recency") >= 90,  2)
         .when(F.col("Recency") >= 45,  3)
         .otherwise(4)
    )
    .withColumn(
        "F_score",
        F.when(F.col("No_of_Visits") > 6, 4)
         .when(F.col("No_of_Visits") > 3, 3)
         .when(F.col("No_of_Visits") > 1, 2)
         .otherwise(1)
    )
    .withColumn(
        "M_score",
        F.when(F.col("Revenue_per_Visit") >= 40, 4)
         .when(F.col("Revenue_per_Visit") >= 30, 3)
         .when(F.col("Revenue_per_Visit") >= 20, 2)
         .otherwise(1)
    )
)

# ---------------------------------------------------------
# 3) CTE_B: RFM_Score as 3-digit string (e.g., "443")
# ---------------------------------------------------------
cte_b = (
    tmp
    .withColumn(
        "RFM_Score",
        F.concat(
            F.col("R_score").cast("string"),
            F.col("F_score").cast("string"),
            F.col("M_score").cast("string"),
        )
    )
)

# ---------------------------------------------------------
# 4) T1 (member_base): Segment mapping from RFM_Score
#    NOTE: comparisons are string-based to match "443" etc.
# ---------------------------------------------------------
champions = ["443", "444"]
potential_loyalist = ["232","233","234","242","243","244","332","333","334","342","343","344","432","433","434","442"]
value_conscious = ["231","241","331","341","431","441"]
promising = ["221","222","223","224","321","322","323","324","421","422","423","424"]
new_seg = ["311","312","313","314","411","412","413","414"]
at_risk = ["121","122","123","124","131","132","133","134","141","142","143","144"]
fleeting = ["111","112","211","212","213","214"]
high_value_lapsed = ["113","114"]

member_base = (  # this is T1 going forward
    cte_b
    .withColumn(
        "Segment",
        F.when(F.col("RFM_Score").isin(champions), "Champions")
         .when(F.col("RFM_Score").isin(potential_loyalist), "Potential Loyalist")
         .when(F.col("RFM_Score").isin(value_conscious), "Value Conscious Loyals")
         .when(F.col("RFM_Score").isin(promising), "Promising")
         .when(F.col("RFM_Score").isin(new_seg), "New")
         .when(F.col("RFM_Score").isin(at_risk), "At Risk")
         .when(F.col("RFM_Score").isin(fleeting), "Fleeting")
         .when(F.col("RFM_Score").isin(high_value_lapsed), "High Value Lapsed")
         .otherwise("NA")
    )
)

t2_recognitions = (
    T_subscriber
    .filter(F.col("snapshot_date_id") == snapshot_date_id)
    .filter(F.col("Renewal_Status").like("Active%"))
    .groupBy("CDE_ID")
    .agg(
        F.sum("Recognitions_Redeemed").alias("Recognitions_Redeemed"),
        F.sum("Member_Priced_Recognitions_Redeemed").alias("Member_Priced_Recognitions_Redeemed"),
        F.sum("Recognitions_Left").alias("Recognitions_Left"),
    )
)

t3_tickets = (
    T_trans
    .filter(F.col("TicketFLAG") == "Yes")
    .filter(F.col("CineClubFLAG") == "Yes")
    .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
    .filter(F.col("datefull").between(F.lit(start_date), F.lit(end_date)))
    .groupBy("CDE_ID")
    .agg(
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 6, F.col("Quantity"))).alias("Member_Tickets"),
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 5, F.col("Quantity"))).alias("Companion_Tickets"),
    )
)


In [0]:

# ----------------------------
# CTE_A: Enriched high-value members
# ----------------------------
cte_a = (
    member_base.alias("t1")   # T1
    .join(t2_recognitions.alias("t2"), on="CDE_ID", how="left")
    .join(t3_tickets.alias("t3"), on="CDE_ID", how="left")
)


In [0]:
from pyspark.sql import functions as F

cte_2025_rfm = (
    cte_a
    .withColumn(
        "Category",
        F.when(F.coalesce(F.col("Recognitions_Redeemed"), F.lit(0)) > 0, F.lit("Yes")).otherwise(F.lit("No"))
    )
    .groupBy("Segment", "Category")
    .agg(
        F.countDistinct("CDE_ID").alias("No_of_Members"),
        F.sum("No_of_Visits").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession_2025"),
        F.sum("Net_Revenue").alias("Net_Revenue_2025"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets_2025"),
        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession_2025"),
        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets_2025"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets_2025"),
        F.sum("Member_Tickets").alias("Member_Tickets"),
        F.sum("Companion_Tickets").alias("Companion_Tickets")
        
    )
)


In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ------------------------------------------------------------
# INPUT: cte_2025_rfm (aggregated by Segment, Category)
# ------------------------------------------------------------

# ----------------------------
# safe divide
# ----------------------------
def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

# ----------------------------
# 1) Normalize column names
# ----------------------------
df_base = (
    cte_2025_rfm
    .select(
        "Segment",
        "Category",
        F.col("No_of_Members").alias("No_of_members"),
        F.col("No_of_Visits_2025").alias("No_of_Visits"),
        F.col("Visits_with_Concession_2025").alias("Visits_with_Concession"),
        F.col("Net_Revenue_2025").alias("Net_Revenue"),
        F.col("Net_Revenue_Tickets_2025").alias("Net_Revenue_Tickets"),
        F.col("Net_Revenue_Concession_2025").alias("Net_Revenue_Concession"),
        F.col("No_of_Regular_Tickets_2025").alias("No_of_Regular_Tickets"),
        F.col("No_of_Premium_Tickets_2025").alias("No_of_Premium_Tickets"),
        F.col("Member_Tickets").alias("Member_Tickets"),
        F.col("Companion_Tickets").alias("Companion_Tickets"),
    )
)

# ----------------------------
# 2) Make 3 populations
#    A) 3 segments individually (keep Segment)
#    B) top3 combined (Segment = "Top 3 Segments")
#    C) all combined (Segment = "All Segments")
# ----------------------------
top3 = ["Champions", "Potential Loyalist", "Value Conscious Loyals"]

df_individual = df_base.filter(F.col("Segment").isin(top3))

df_top3 = (
    df_base.filter(F.col("Segment").isin(top3))
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),
        F.sum("No_of_Visits").alias("No_of_Visits"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession"),
        F.sum("Net_Revenue").alias("Net_Revenue"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets"),
        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession"),
        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets"),
        F.sum("Member_Tickets").alias("Member_Tickets"),
        F.sum("Companion_Tickets").alias("Companion_Tickets"),
    )
    .withColumn("Segment", F.lit("Top 3 Segments"))
)

df_all = (
    df_base
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),
        F.sum("No_of_Visits").alias("No_of_Visits"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession"),
        F.sum("Net_Revenue").alias("Net_Revenue"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets"),
        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession"),
        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets"),
        F.sum("Member_Tickets").alias("Member_Tickets"),
        F.sum("Companion_Tickets").alias("Companion_Tickets"),
    )
    .withColumn("Segment", F.lit("All Segments"))
)

# union all populations into one frame to compute metrics consistently
df_pop = df_individual.unionByName(df_top3, allowMissingColumns=True).unionByName(df_all, allowMissingColumns=True)

# ----------------------------
# 3) Derived metrics (per Segment x Category)
# ----------------------------
df_calc = (
    df_pop
    .withColumn("Avg Frequency", sdiv(F.col("No_of_Visits"), F.col("No_of_members")))
    .withColumn("Avg Total Spend per visit", sdiv(F.col("Net_Revenue"), F.col("No_of_Visits")))
    .withColumn("Avg Ticket Spend per visit", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Avg Concession Spend per visit", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Spend per person in a year", sdiv(F.col("Net_Revenue"), F.col("No_of_members")))
    .withColumn("Ticket Spend per person in a year", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Spend per person in a year", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Incidence", sdiv(F.col("Visits_with_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Tickets per Visit", sdiv(F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets"), F.col("No_of_Visits")))
    .withColumn("999 Tickets per Visit", sdiv(F.col("Member_Tickets") + F.col("Companion_Tickets"), F.col("No_of_Visits")))
    .withColumn(
        "Uptake Premium Experiences",
        sdiv(F.col("No_of_Premium_Tickets"), (F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets")))
    )
    .withColumn(
        "Uptake Companion Experiences",
        sdiv(F.col("Companion_Tickets"), F.col("Member_Tickets"))
    )
)

# ----------------------------
# 4) Reshape to "metrics as rows" PER SEGMENT
# ----------------------------
metric_order = [
    "No_of_members",
    "No_of_Visits",
    "Visits_with_Concession",
    "Net_Revenue",
    "Net_Revenue_Concession",
    "No_of_Regular_Tickets",
    "No_of_Premium_Tickets",
    "Member_Tickets",
    "Companion_Tickets",
    "Avg Frequency",
    "Avg Total Spend per visit",
    "Avg Ticket Spend per visit",
    "Avg Concession Spend per visit",
    "Total Spend per person in a year",
    "Ticket Spend per person in a year",
    "Concession Spend per person in a year",
    "Concession Incidence",
    "Total Tickets per Visit",
    "999 Tickets per Visit",
    "Uptake Premium Experiences",
    "Uptake Companion Experiences"
]

df_long = reduce(
    lambda a, b: a.unionByName(b),
    [
        df_calc.select("Segment", "Category", F.lit(m).alias("Metric"), F.col(m).alias("Value"))
        for m in metric_order
    ]
)

df_out = (
    df_long
    .groupBy("Segment", "Metric")
    .pivot("Category", ["Yes", "No"])
    .agg(F.first("Value"))
)

# enforce metric row order (within each Segment)
order_map = F.create_map(*[
    x for kv in [(m, i) for i, m in enumerate(metric_order, start=1)]
    for x in (F.lit(kv[0]), F.lit(kv[1]))
])

# order segments in your preferred sequence
segment_sort = F.when(F.col("Segment") == "Champions", 1) \
    .when(F.col("Segment") == "Potential Loyalist", 2) \
    .when(F.col("Segment") == "Value Conscious Loyals", 3) \
    .when(F.col("Segment") == "Top 3 Segments", 4) \
    .when(F.col("Segment") == "All Segments", 5) \
    .otherwise(999)

df_out = (
    df_out
    .withColumn("_seg_order", segment_sort)
    .withColumn("_metric_order", order_map[F.col("Metric")])
    .orderBy("_seg_order", "_metric_order")
    .drop("_seg_order", "_metric_order")
)

In [0]:
df_out.display()

In [0]:
cte_2025_rfm_member = (
    cte_a
    .withColumn(
        "Category",
        F.when(F.coalesce(F.col("Recognitions_Redeemed"), F.lit(0)) > 0, F.lit("Yes")).otherwise(F.lit("No"))
    )
    .groupBy("t1.CDE_ID","Category","Segment")
    .agg(
        F.countDistinct("CDE_ID").alias("No_of_Members"),
        F.sum("No_of_Visits").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession_2025"),
        F.sum("Net_Revenue").alias("Net_Revenue_2025"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets_2025"),
        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession_2025"),
        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets_2025"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets_2025"),
        F.sum("Member_Tickets").alias("Member_Tickets"),
        F.sum("Companion_Tickets").alias("Companion_Tickets")
        
    )
)


In [0]:
from pyspark.sql import functions as F

import pandas as pd
from pyspark.sql import functions as F

# ============================================================
# 1) Load the Excel file via pandas and convert to Spark
# ============================================================
creation_dates_pdf = pd.read_excel(
    "/Volumes/data_exploration_hub/marketing_hub/offline_data/Creation Dates.xlsx"
)

# 🚨 Remove specific user
creation_dates_pdf = creation_dates_pdf[
    creation_dates_pdf["cc_user_profile_id"] != "AzqBunUAsPVB41nKT"
]

creation_dates_df = spark.createDataFrame(creation_dates_pdf)

# Join cc_user_profile_id from Excel to CDE_ID via T_subscriber
cc_join_dates = (
    creation_dates_df
    .join(
        T_subscriber.select("cc_user_profile_id", "CDE_ID").distinct(),
        on="cc_user_profile_id",
        how="inner"
    )
    .select(
        F.col("CDE_ID"),
        F.to_date(F.col("customers_created_at")).alias("join_date")
    )
    .distinct()
)

# Members who joined CineClub before 2025 — to be excluded
joined_before_2025 = (
    cc_join_dates
    .filter(F.col("join_date") < F.lit("2025-01-01"))
    .select(F.col("CDE_ID").alias("early_joiner_ID"))
    .distinct()
)


In [0]:
# Members who joined CineClub before 2025 — to be excluded
joined_in_2025 = (
    cc_join_dates
    .filter(F.col("join_date") > F.lit("2025-01-01"))
    .select(F.col("CDE_ID"))
    .distinct()
)

In [0]:
# joined_in_2025.display()

In [0]:
cte_2023 = (
    T_trans.alias("t")
    .join(T_locations.alias("l"), F.col("t.LocationId") == F.col("l.LocationID"), "left")
    .join(T_date.alias("d"), F.col("t.VisitDateId") == F.col("d.dateid"), "left")
    
    # 🚨 EXCLUDE early joiners
    .join(joined_before_2025, F.col("t.CDE_ID") == F.col("early_joiner_ID"), "left_anti")
    
    .filter(F.col("t.CDE_ID").isNotNull())
    .withColumn("CDE_Visit_ID_MNL", F.concat_ws("|",
        F.col("t.CDE_ID").cast("string"),
        F.col("t.VisitDateId").cast("string"),
        F.col("t.LocationId").cast("string")
    ))
    .filter(F.col("d.datefull").between(F.lit(start_2023), F.lit(end_2023)))
    .filter(F.col("l.LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("t.SCENEFLAG") == "Yes")
    .groupBy(F.col("t.CDE_ID"))
    .agg(
        F.countDistinct(F.when(F.col("t.Net_Revenue") > 0, F.col("CDE_Visit_ID_MNL"))).alias("No_of_Visits_2023"),
        F.countDistinct(
            F.when((F.col("t.Net_Revenue") > 0) & (F.col("t.ConcessionFLAG") == "Yes"), F.col("CDE_Visit_ID_MNL"))
        ).alias("Visits_with_Concession_2023"),
        F.sum(F.col("t.Net_Revenue")).alias("Revenue_2023"),
        F.sum(F.when(F.col("t.ConcessionFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(0)).alias("Rev_Conc_2023"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(0)).alias("Rev_Tickets_2023"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Quantity")).otherwise(0)).alias("Tickets_2023"),
    )
    .filter(F.col("Revenue_2023") > 0)
    .withColumnRenamed("CDE_ID", "ID_2023")
)
cte_2024 = (
    T_trans.alias("t")
    .join(T_locations.alias("l"), F.col("t.LocationId") == F.col("l.LocationID"), "left")
    .join(T_date.alias("d"), F.col("t.VisitDateId") == F.col("d.dateid"), "left")
    
    # 🚨 EXCLUDE early joiners
    .join(joined_before_2025, F.col("t.CDE_ID") == F.col("early_joiner_ID"), "left_anti")
    
    .filter(F.col("t.CDE_ID").isNotNull())
    .withColumn("CDE_Visit_ID_MNL", F.concat_ws("|",
        F.col("t.CDE_ID").cast("string"),
        F.col("t.VisitDateId").cast("string"),
        F.col("t.LocationId").cast("string")
    ))
    .filter(F.col("d.datefull").between(F.lit(start_2024), F.lit(end_2024)))
    .filter(F.col("l.LC_Location_Type_Description") == "Cineplex Theatre")
    .filter(F.col("t.SCENEFLAG") == "Yes")
    .groupBy(F.col("t.CDE_ID"))
    .agg(
        F.countDistinct(F.when(F.col("t.Net_Revenue") > 0, F.col("CDE_Visit_ID_MNL"))).alias("No_of_Visits_2024"),
        F.countDistinct(
            F.when((F.col("t.Net_Revenue") > 0) & (F.col("t.ConcessionFLAG") == "Yes"), F.col("CDE_Visit_ID_MNL"))
        ).alias("Visits_with_Concession_2024"),
        F.sum(F.col("t.Net_Revenue")).alias("Revenue_2024"),
        F.sum(F.when(F.col("t.ConcessionFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(0)).alias("Rev_Conc_2024"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(0)).alias("Rev_Tickets_2024"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Quantity")).otherwise(0)).alias("Tickets_2024"),
    )
    .filter(F.col("Revenue_2024") > 0)
    .withColumnRenamed("CDE_ID", "ID_2024")
)

cte_2025 = (
    T_trans.alias("t")
    .join(T_locations.alias("l"), F.col("t.LocationId") == F.col("l.LocationID"), "left")
    .join(T_date.alias("d"), F.col("t.VisitDateId") == F.col("d.dateid"), "left")
    .withColumn("CDE_Visit_ID_MNL", F.concat_ws("|",
        F.col("t.CDE_ID").cast("string"),
        F.col("t.VisitDateId").cast("string"),
        F.col("t.LocationId").cast("string")
            )
        )
    .filter(F.col("t.CDE_ID").isNotNull())
    .filter(F.col("d.datefull").between(F.lit(start_date), F.lit(end_date)))
    .filter(F.col("l.LC_Location_Type_Description") == F.lit("Cineplex Theatre"))
    .filter(F.col("t.SCENEFLAG") == "Yes")
    .groupBy(F.col("t.CDE_ID"))
    .agg(
        F.countDistinct(F.when(F.col("t.Net_Revenue") > 0, F.col("CDE_Visit_ID_MNL"))).alias("No_of_Visits_2025"),
        F.countDistinct(
            F.when((F.col("t.Net_Revenue") > 0) & (F.col("t.ConcessionFLAG") == "Yes"), F.col("CDE_Visit_ID_MNL"))
        ).alias("Visits_with_Concession_2025"),
        F.sum(F.col("t.Net_Revenue")).alias("Revenue_2025"),
        F.sum(F.when(F.col("t.ConcessionFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(F.lit(0))).alias("Rev_Conc_2025"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Net_Revenue")).otherwise(F.lit(0))).alias("Rev_Tickets_2025"),
        F.sum(F.when(F.col("t.TicketFLAG") == "Yes", F.col("t.Quantity")).otherwise(F.lit(0))).alias("Tickets_2025"),
    )
    .filter(F.col("Revenue_2025") > 0)
    .withColumnRenamed("CDE_ID", "ID_2025")
)

# ============================================================
# 3) tmp5: join CTE_2025_RFM with CTE_2024 + CTE_2025, Category = 'No'
#     Assumes you already built cte_2025_rfm with columns: CDE_ID, Segment, Category, ...
# ============================================================
# tmp5 = (
#     cte_2025_rfm_member.alias("r")
#     .join(cte_2023.alias("y23"), F.col("y23.ID_2023") == F.col("r.CDE_ID"), "inner")
#     .join(cte_2024.alias("y19"), F.col("y19.ID_2024") == F.col("r.CDE_ID"), "inner")
#     .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("r.CDE_ID"), "inner")
#     .select(
#         F.col("r.CDE_ID").alias("final_CDE_ID"),
#         F.col("r.Segment"),
#         F.col("y23.*"),
#         F.col("y25.*"),
#         F.col("y19.*"),
#         F.col("r.Category")
#     )
# )

# tmp5.cache()

In [0]:
tmp5 = (
    cte_2025_rfm_member.alias("r")
    .join(cte_2023.alias("y23"), F.col("y23.ID_2023") == F.col("r.CDE_ID"), "inner")
    .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("r.CDE_ID"), "inner")
    .select(
        F.col("r.CDE_ID").alias("final_CDE_ID"),
        F.col("r.Segment"),
        F.col("y23.*"),
        F.col("y25.*"),
        F.col("r.Category")
    )
)

tmp5.cache()

In [0]:
tmp10 = (
    cte_2025_rfm_member.alias("r")
    .join(cte_2024.alias("y24"), F.col("y24.ID_2024") == F.col("r.CDE_ID"), "inner")
    .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("r.CDE_ID"), "inner")
    .select(
        F.col("r.CDE_ID").alias("final_CDE_ID"),
        F.col("r.Segment"),
        F.col("y24.*"),
        F.col("y25.*"),
        F.col("r.Category")
    )
)

In [0]:
from pyspark.sql import functions as F

df_cc_all_segments = (
    tmp5
    .filter(F.col("Category") == "Yes")   # CC = Yes
)


# display(df_cc_all_segments)

In [0]:
df_questions = (
    df_cc_all_segments
    .select("final_CDE_ID")
    .join(T_subscriber.alias("s"), F.col("s.CDE_ID") == F.col("final_CDE_ID"), "left")
    .filter(F.col("s.snapshot_date_id") == F.lit("20251231"))
    .select("s.CDE_ID", "s.plan_id", "s.renewal_status")
)

In [0]:
from pyspark.sql import functions as F

# =============================================================================
# STEP 1: Define the ORIGINAL base cohort (users active as of Dec 31, 2025)
# =============================================================================

base_cohort = (
    df_cc_all_segments
    .select("final_CDE_ID")
    .join(T_subscriber.alias("s"), F.col("s.CDE_ID") == F.col("final_CDE_ID"), "left")
    .filter(F.col("s.snapshot_date_id") == F.lit("20251231"))
    .filter(F.col("s.renewal_status").like("Active%"))
    .select(F.col("s.CDE_ID").alias("cohort_CDE_ID"))
    .distinct()
)

base_cohort.cache()
original_cohort_size = base_cohort.count()
print(f"Original cohort size (Dec 31, 2025): {original_cohort_size}")

# =============================================================================
# STEP 2: Define month boundaries
# =============================================================================

months = [
    ("January",   "20260131", "20251231"),
    ("February",  "20260228", "20260131"),
    ("March",     "20260331", "20260228"),
]
# Format: (month_name, month_end, prior_month_end)

# =============================================================================
# STEP 3: For each month, from ORIGINAL cohort:
#         - How many were active at month start?
#         - Of those, how many churned (not active at month end)?
# =============================================================================

results = []

for month_name, month_end, prior_month_end in months:
    
    # From ORIGINAL cohort: who was active at the START of this month?
    active_at_month_start = (
        base_cohort
        .join(
            T_subscriber.alias("s"),
            F.col("s.CDE_ID") == F.col("cohort_CDE_ID"),
            "inner"
        )
        .filter(F.col("s.snapshot_date_id") == F.lit(prior_month_end))
        .filter(F.col("s.renewal_status").like("Active%"))
        .select(F.col("cohort_CDE_ID").alias("active_CDE_ID"))
        .distinct()
    )
    
    # Of those active at month start: who was still active at month end?
    still_active_at_month_end = (
        active_at_month_start
        .join(
            T_subscriber.alias("s"),
            F.col("s.CDE_ID") == F.col("active_CDE_ID"),
            "inner"
        )
        .filter(F.col("s.snapshot_date_id") == F.lit(month_end))
        .filter(F.col("s.renewal_status").like("Active%"))
        .select("active_CDE_ID")
        .distinct()
    )
    
    # Churned = active at month start but NOT active at month end
    churned_this_month = (
        active_at_month_start
        .join(still_active_at_month_end, 
              active_at_month_start["active_CDE_ID"] == still_active_at_month_end["active_CDE_ID"], 
              "left_anti")
    )
    
    # Counts
    active_start_count = active_at_month_start.count()
    churned_count = churned_this_month.count()
    churn_rate = (churned_count / active_start_count * 100) if active_start_count > 0 else 0
    
    results.append({
        "month": month_name,
        "original_cohort": original_cohort_size,
        "active_at_month_start": active_start_count,
        "churned_this_month": churned_count,
        "churn_rate_pct": round(churn_rate, 2)
    })

# =============================================================================
# STEP 4: Create summary DataFrame
# =============================================================================

df_churn_summary = spark.createDataFrame(results)
df_churn_summary.show()

In [0]:
df_churn_summary.display()

In [0]:
df_questions = (
    df_cc_all_segments
    .select("ID_2025")
    .join(T_subscriber.alias("s"), F.col("s.CDE_ID") == F.col("ID_2025"), "left")
    .filter(F.col("s.snapshot_date_id") == F.lit("20260401"))
    .select("s.CDE_ID", "s.plan_id", "s.renewal_status")
)

In [0]:
df_questions.display()

In [0]:
# # Exclusion set: anyone who was a CineClub member in 2023
# cineclub_2023_ids = (
#     T_trans
#     .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
#     .filter(F.col("CineClubFLAG") == "Yes")
#     .filter(F.col("datefull").between(F.lit(start_2023), F.lit(end_2023)))
#     .select(F.col("CDE_ID").alias("cc_2023_ID"))
#     .distinct()
# )

# # Exclusion set: anyone who was a CineClub member in 2023
# cineclub_2024_ids = (
#     T_trans
#     .join(T_date, F.col("VisitDateId") == F.col("dateid"), "left")
#     .filter(F.col("CineClubFLAG") == "Yes")
#     .filter(F.col("datefull").between(F.lit(start_2024), F.lit(end_2024)))
#     .select(F.col("CDE_ID").alias("cc_2024_ID"))
#     .distinct()
# )

# tmp5 = (
#     cte_2025_rfm_member.alias("r")
#     .join(cte_2024.alias("y24"), F.col("y24.ID_2024") == F.col("r.CDE_ID"), "inner")
#     .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("r.CDE_ID"), "inner")
#     # Exclude anyone who had CineClubFLAG = 'Yes' on any 2023 transaction
#     .join(cineclub_2024_ids, F.col("r.CDE_ID") == F.col("cc_2024_ID"), "left_anti")
#     .select(
#         F.col("r.CDE_ID").alias("final_CDE_ID"),
#         F.col("r.Segment"),
#         F.col("y24.*"),
#         F.col("y25.*"),
#         F.col("r.Category")
#     )
# )

# tmp5.cache()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

top3 = ["Champions", "Potential Loyalist", "Value Conscious Loyals"]

metric_order_labels = [
    "No. of Members",
    "No. of Visits",
    "Visits with Concession",
    "Revenue",
    "Rev_Conc",
    "No of Tickets",
    "Average Frequency",
    "Avg Total Spend per Visit",
    "Avg Ticket Spend per Visit",
    "Avg Concession Spend per visit",
    "Total Spend per person in a year",
    "Ticket Spend per person in a year",
    "Concession Spend per person in a year",
    "Concession Incidence",
    "Total Tickets per Visit"
]

# ============================================================
# 1) Base at Segment x Category
# ============================================================
df_base_seg = (
    tmp5
    .groupBy("Segment", "Category")
    .agg(
        F.count("final_CDE_ID").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
)

# ============================================================
# 2) Populations: individuals + Top3 combined + All combined
# ============================================================
df_individual = df_base_seg.filter(F.col("Segment").isin(top3))

df_top3 = (
    df_base_seg.filter(F.col("Segment").isin(top3))
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
    .withColumn("Segment", F.lit("Top 3 Segments"))
)

df_all = (
    df_base_seg
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
    .withColumn("Segment", F.lit("All Segments"))
)

df_pop = df_individual.unionByName(df_top3).unionByName(df_all)

# ============================================================
# 3) Derived metrics PER YEAR
# ============================================================
df_calc = (
    df_pop

    # ---- 2023 ----
    .withColumn("Average_Frequency_2023", sdiv(F.col("No_of_Visits_2023"), F.col("No_of_members")))
    .withColumn("Avg_Total_Spend_per_Visit_2023", sdiv(F.col("Revenue_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Avg_Ticket_Spend_per_Visit_2023", sdiv(F.col("Revenue_2023") - F.col("Rev_Conc_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Avg_Concession_Spend_per_Visit_2023", sdiv(F.col("Rev_Conc_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Total_Spend_per_Person_per_Year_2023", sdiv(F.col("Revenue_2023"), F.col("No_of_members")))
    .withColumn("Ticket_Spend_per_Person_per_Year_2023", sdiv(F.col("Revenue_2023") - F.col("Rev_Conc_2023"), F.col("No_of_members")))
    .withColumn("Concession_Spend_per_Person_per_Year_2023", sdiv(F.col("Rev_Conc_2023"), F.col("No_of_members")))
    .withColumn("Concession_Incidence_2023", sdiv(F.col("Visits_with_Concession_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Total_Tickets_per_Visit_2023", sdiv(F.col("Tickets_2023"), F.col("No_of_Visits_2023")))

    # ---- 2025 ----
    .withColumn("Average_Frequency_2025", sdiv(F.col("No_of_Visits_2025"), F.col("No_of_members")))
    .withColumn("Avg_Total_Spend_per_Visit_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg_Ticket_Spend_per_Visit_2025", sdiv(F.col("Revenue_2025") - F.col("Rev_Conc_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg_Concession_Spend_per_Visit_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Total_Spend_per_Person_per_Year_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_members")))
    .withColumn("Ticket_Spend_per_Person_per_Year_2025", sdiv(F.col("Revenue_2025") - F.col("Rev_Conc_2025"), F.col("No_of_members")))
    .withColumn("Concession_Spend_per_Person_per_Year_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_members")))
    .withColumn("Concession_Incidence_2025", sdiv(F.col("Visits_with_Concession_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Total_Tickets_per_Visit_2025", sdiv(F.col("Tickets_2025"), F.col("No_of_Visits_2025")))
)

# ============================================================
# 4) Final output: Segment | Category | Metric | Year_2023 | Year_2025
# ============================================================
metric_map = [
    ("No. of Members",                          "No_of_members",                          "No_of_members"),
    ("No. of Visits",                           "No_of_Visits_2023",                      "No_of_Visits_2025"),
    ("Visits with Concession",                  "Visits_with_Concession_2023",            "Visits_with_Concession_2025"),
    ("Revenue",                                 "Revenue_2023",                           "Revenue_2025"),
    ("Rev_Conc",                                "Rev_Conc_2023",                          "Rev_Conc_2025"),
    ("No of Tickets",                           "Tickets_2023",                           "Tickets_2025"),
    ("Average Frequency",                       "Average_Frequency_2023",                 "Average_Frequency_2025"),
    ("Avg Total Spend per Visit",               "Avg_Total_Spend_per_Visit_2023",         "Avg_Total_Spend_per_Visit_2025"),
    ("Avg Ticket Spend per Visit",              "Avg_Ticket_Spend_per_Visit_2023",        "Avg_Ticket_Spend_per_Visit_2025"),
    ("Avg Concession Spend per visit",          "Avg_Concession_Spend_per_Visit_2023",    "Avg_Concession_Spend_per_Visit_2025"),
    ("Total Spend per person in a year",        "Total_Spend_per_Person_per_Year_2023",   "Total_Spend_per_Person_per_Year_2025"),
    ("Ticket Spend per person in a year",       "Ticket_Spend_per_Person_per_Year_2023",  "Ticket_Spend_per_Person_per_Year_2025"),
    ("Concession Spend per person in a year",   "Concession_Spend_per_Person_per_Year_2023", "Concession_Spend_per_Person_per_Year_2025"),
    ("Concession Incidence",                    "Concession_Incidence_2023",              "Concession_Incidence_2025"),
    ("Total Tickets per Visit",                 "Total_Tickets_per_Visit_2023",           "Total_Tickets_per_Visit_2025"),
]

df_out = reduce(
    lambda a, b: a.unionByName(b),
    [
        df_calc.select(
            "Segment",
            "Category",
            F.lit(metric_label).alias("Metric"),
            F.col(col_2023).alias("Year_2023"),
            F.col(col_2025).alias("Year_2025"),
        )
        for (metric_label, col_2023, col_2025) in metric_map
    ]
)

seg_order = (
    F.when(F.col("Segment") == "Champions", 1)
     .when(F.col("Segment") == "Potential Loyalist", 2)
     .when(F.col("Segment") == "Value Conscious Loyals", 3)
     .when(F.col("Segment") == "Top 3 Segments", 4)
     .when(F.col("Segment") == "All Segments", 5)
     .otherwise(999)
)

cat_order = (
    F.when(F.col("Category") == "No", 1)
     .when(F.col("Category") == "Yes", 2)
     .otherwise(999)
)

order_map = F.create_map(*[
    x for kv in [(m, i) for i, m in enumerate(metric_order_labels, start=1)]
    for x in (F.lit(kv[0]), F.lit(kv[1]))
])

df_out = (
    df_out
    .withColumn("_seg_order", seg_order)
    .withColumn("_cat_order", cat_order)
    .withColumn("_metric_order", order_map[F.col("Metric")])
    .orderBy("_seg_order", "_cat_order", "_metric_order")
    .drop("_seg_order", "_cat_order", "_metric_order")
)

In [0]:
df_out.display()

In [0]:
from pyspark.sql import functions as F

# ── Specify inflation factors directly ──────────────────────────────────────
TICKET_INFLATION     = 0.062   # 6.2%
CONCESSION_INFLATION = 0.095   # 9.5%

# ── Pivot df_out to one row per Segment, CineClub only ──────────────────────
metrics_needed = [
    "No. of Members",
    "Average Frequency",
    "Avg Ticket Spend per Visit",
    "Avg Concession Spend per visit",
]

df_wide = (
    df_out
    .filter(
        (F.col("Category") == "Yes") &
        (F.col("Metric").isin(metrics_needed))
    )
    .groupBy("Segment")
    .pivot("Metric", metrics_needed)
    .agg(
        F.first(
            F.struct(
                F.col("Year_2023").alias("y2023"),
                F.col("Year_2025").alias("y2025"),
            )
        )
    )
)

df_wide = (
    df_wide
    .select(
        "Segment",
        F.col("`No. of Members`.y2023").alias("No_of_Members"),
        F.col("`Average Frequency`.y2023").alias("Freq_2023"),
        F.col("`Average Frequency`.y2025").alias("Freq_2025"),
        F.col("`Avg Ticket Spend per Visit`.y2023").alias("Ticket_Spend_2023"),
        F.col("`Avg Ticket Spend per Visit`.y2025").alias("Ticket_Spend_2025_Actual"),
        F.col("`Avg Concession Spend per visit`.y2023").alias("Conc_Spend_2023"),
        F.col("`Avg Concession Spend per visit`.y2025").alias("Conc_Spend_2025_Actual"),
    )
)

def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

df_incremental = (
    df_wide
    .withColumn("Ticket_2025_Inflation_Adj",
                F.col("Ticket_Spend_2023") * F.lit(1 + TICKET_INFLATION))
    .withColumn("Conc_2025_Inflation_Adj",
                F.col("Conc_Spend_2023") * F.lit(1 + CONCESSION_INFLATION))
    .withColumn("Ticket_Pct_Incremental_Spend_per_Visit",
                sdiv(
                    F.col("Ticket_Spend_2025_Actual") - F.col("Ticket_2025_Inflation_Adj"),
                    F.col("Ticket_2025_Inflation_Adj")
                ))
    .withColumn("Conc_Pct_Incremental_Spend_per_Visit",
                sdiv(
                    F.col("Conc_Spend_2025_Actual") - F.col("Conc_2025_Inflation_Adj"),
                    F.col("Conc_2025_Inflation_Adj")
                ))
    .withColumn("Incremental_Visits",
                F.col("Freq_2025") - F.col("Freq_2023"))
    .withColumn("Ticket_Incremental_Spend_per_Member",
                (F.col("Ticket_Spend_2025_Actual") - F.col("Ticket_2025_Inflation_Adj"))
                    * F.col("Freq_2025")
                + F.col("Ticket_2025_Inflation_Adj") * F.col("Incremental_Visits"))
    .withColumn("Conc_Incremental_Spend_per_Member",
                (F.col("Conc_Spend_2025_Actual") - F.col("Conc_2025_Inflation_Adj"))
                    * F.col("Freq_2025")
                + F.col("Conc_2025_Inflation_Adj") * F.col("Incremental_Visits"))
    .withColumn("Ticket_Total_Incremental_Spend",
                F.col("Ticket_Incremental_Spend_per_Member") * F.col("No_of_Members"))
    .withColumn("Conc_Total_Incremental_Spend",
                F.col("Conc_Incremental_Spend_per_Member") * F.col("No_of_Members"))
)

seg_order = (
    F.when(F.col("Segment") == "Champions",              1)
     .when(F.col("Segment") == "Potential Loyalist",     2)
     .when(F.col("Segment") == "Value Conscious Loyals", 3)
     .when(F.col("Segment") == "Top 3 Segments",         4)
     .when(F.col("Segment") == "All Segments",           5)
     .otherwise(999)
)

df_result = (
    df_incremental
    .select(
        "Segment",
        F.col("No_of_Members").cast("long"),
        F.round("Ticket_Spend_2023",                     2).alias("Ticket_2023_Actual"),
        F.round("Ticket_2025_Inflation_Adj",             2).alias("Ticket_2025_Inflation_Adj"),
        F.round("Ticket_Spend_2025_Actual",              2).alias("Ticket_2025_Actual"),
        F.round("Ticket_Pct_Incremental_Spend_per_Visit",4).alias("Ticket_Pct_Incremental_Spend_per_Visit"),
        F.round("Incremental_Visits",                    2).alias("Incremental_Visits"),
        F.round("Ticket_Incremental_Spend_per_Member",   2).alias("Ticket_Incremental_Spend_per_Member"),
        F.round("Ticket_Total_Incremental_Spend",        0).alias("Ticket_Total_Incremental_Spend"),
        F.round("Conc_Spend_2023",                       2).alias("Conc_2023_Actual"),
        F.round("Conc_2025_Inflation_Adj",               2).alias("Conc_2025_Inflation_Adj"),
        F.round("Conc_Spend_2025_Actual",                2).alias("Conc_2025_Actual"),
        F.round("Conc_Pct_Incremental_Spend_per_Visit",  4).alias("Conc_Pct_Incremental_Spend_per_Visit"),
        F.round("Incremental_Visits",                    2).alias("Incremental_Visits_Conc"),
        F.round("Conc_Incremental_Spend_per_Member",     2).alias("Conc_Incremental_Spend_per_Member"),
        F.round("Conc_Total_Incremental_Spend",          0).alias("Conc_Total_Incremental_Spend"),
    )
    .withColumn("_seg_order", seg_order)
    .orderBy("_seg_order")
    .drop("_seg_order")
)


In [0]:
df_result.display()

In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ============================================================
# FULL REPLACEMENT BLOCK (2024 + 2025)
# Output columns:
#   Segment | Category | Metric | Year_2024 | Year_2025
# Populations:
#   1) Champions, Potential Loyalist, Value Conscious Loyals (individually)
#   2) Top 3 Segments (combined)
#   3) All Segments (combined)
# Metric order: (your exact list)
# ============================================================

# ----------------------------
# safe divide
# ----------------------------
def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

top3 = ["Champions", "Potential Loyalist", "Value Conscious Loyals"]

# ----------------------------
# Metric order + mapping (display label -> (2024 col, 2025 col))
# ----------------------------
metric_order_labels = [
    "No. of Members",
    "No. of Visits",
    "Visits with Concession",
    "Revenue",
    "Rev_Conc",
    "No of Tickets",
    "Average Frequency",
    "Avg Total Spend per Visit",
    "Avg Ticket Spend per Visit",
    "Avg Concession Spend per visit",
    "Total Spend per person in a year",
    "Ticket Spend per person in a year",
    "Concession Spend per person in a year",
    "Concession Incidence",
    "Total Tickets per Visit"
]

# ============================================================
# 1) Base at Segment x Category
# ============================================================
df_base_seg = (
    tmp5
    .groupBy("Segment", "Category")
    .agg(
        F.count("final_CDE_ID").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("Revenue_2024").alias("Revenue_2024"),
        F.sum("Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("Tickets_2024").alias("Tickets_2024"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
)

# ============================================================
# 2) Populations: individuals + Top3 combined + All combined
# ============================================================
df_individual = df_base_seg.filter(F.col("Segment").isin(top3))

df_top3 = (
    df_base_seg.filter(F.col("Segment").isin(top3))
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("Revenue_2024").alias("Revenue_2024"),
        F.sum("Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("Tickets_2024").alias("Tickets_2024"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
    .withColumn("Segment", F.lit("Top 3 Segments"))
)

df_all = (
    df_base_seg
    .groupBy("Category")
    .agg(
        F.sum("No_of_members").alias("No_of_members"),

        F.sum("No_of_Visits_2023").alias("No_of_Visits_2023"),
        F.sum("Visits_with_Concession_2023").alias("Visits_with_Concession_2023"),
        F.sum("Revenue_2023").alias("Revenue_2023"),
        F.sum("Rev_Conc_2023").alias("Rev_Conc_2023"),
        F.sum("Rev_Tickets_2023").alias("Rev_Tickets_2023"),
        F.sum("Tickets_2023").alias("Tickets_2023"),

        F.sum("No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("Revenue_2024").alias("Revenue_2024"),
        F.sum("Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("Tickets_2024").alias("Tickets_2024"),

        F.sum("No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("Revenue_2025").alias("Revenue_2025"),
        F.sum("Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("Tickets_2025").alias("Tickets_2025"),
    )
    .withColumn("Segment", F.lit("All Segments"))
)

df_pop = df_individual.unionByName(df_top3).unionByName(df_all)

# ============================================================
# 3) Derived metrics PER YEAR
# ============================================================
df_calc = (
    df_pop

    # ---- 2023 ----
    .withColumn("Average_Frequency_2023", sdiv(F.col("No_of_Visits_2023"), F.col("No_of_members")))
    .withColumn("Avg_Total_Spend_per_Visit_2023", sdiv(F.col("Revenue_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Avg_Ticket_Spend_per_Visit_2023", sdiv(F.col("Revenue_2023") - F.col("Rev_Conc_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Avg_Concession_Spend_per_Visit_2023", sdiv(F.col("Rev_Conc_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Total_Spend_per_Person_per_Year_2023", sdiv(F.col("Revenue_2023"), F.col("No_of_members")))
    .withColumn("Ticket_Spend_per_Person_per_Year_2023", sdiv(F.col("Revenue_2023") - F.col("Rev_Conc_2023"), F.col("No_of_members")))
    .withColumn("Concession_Spend_per_Person_per_Year_2023", sdiv(F.col("Rev_Conc_2023"), F.col("No_of_members")))
    .withColumn("Concession_Incidence_2023", sdiv(F.col("Visits_with_Concession_2023"), F.col("No_of_Visits_2023")))
    .withColumn("Total_Tickets_per_Visit_2023", sdiv(F.col("Tickets_2023"), F.col("No_of_Visits_2023")))


    # ---- 2024 ----
    .withColumn("Average_Frequency_2024", sdiv(F.col("No_of_Visits_2024"), F.col("No_of_members")))
    .withColumn("Avg_Total_Spend_per_Visit_2024", sdiv(F.col("Revenue_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Avg_Ticket_Spend_per_Visit_2024", sdiv(F.col("Revenue_2024") - F.col("Rev_Conc_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Avg_Concession_Spend_per_Visit_2024", sdiv(F.col("Rev_Conc_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Total_Spend_per_Person_per_Year_2024", sdiv(F.col("Revenue_2024"), F.col("No_of_members")))
    .withColumn("Ticket_Spend_per_Person_per_Year_2024", sdiv(F.col("Revenue_2024") - F.col("Rev_Conc_2024"), F.col("No_of_members")))
    .withColumn("Concession_Spend_per_Person_per_Year_2024", sdiv(F.col("Rev_Conc_2024"), F.col("No_of_members")))
    .withColumn("Concession_Incidence_2024", sdiv(F.col("Visits_with_Concession_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Total_Tickets_per_Visit_2024", sdiv(F.col("Tickets_2024"), F.col("No_of_Visits_2024")))

    # ---- 2025 ----
    .withColumn("Average_Frequency_2025", sdiv(F.col("No_of_Visits_2025"), F.col("No_of_members")))
    .withColumn("Avg_Total_Spend_per_Visit_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg_Ticket_Spend_per_Visit_2025", sdiv(F.col("Revenue_2025") - F.col("Rev_Conc_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg_Concession_Spend_per_Visit_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Total_Spend_per_Person_per_Year_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_members")))
    .withColumn("Ticket_Spend_per_Person_per_Year_2025", sdiv(F.col("Revenue_2025") - F.col("Rev_Conc_2025"), F.col("No_of_members")))
    .withColumn("Concession_Spend_per_Person_per_Year_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_members")))
    .withColumn("Concession_Incidence_2025", sdiv(F.col("Visits_with_Concession_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Total_Tickets_per_Visit_2025", sdiv(F.col("Tickets_2025"), F.col("No_of_Visits_2025")))
)

# ============================================================
# 4) Final output table: Segment | Category | Metric | Year_2024 | Year_2025
# ============================================================
metric_map = [
    ("No. of Members", "No_of_members", "No_of_members", "No_of_members"),
    ("No. of Visits","No_of_Visits_2023" ,"No_of_Visits_2024", "No_of_Visits_2025"),
    ("Visits with Concession","Visits_with_Concession_2023" ,"Visits_with_Concession_2024", "Visits_with_Concession_2025"),
    ("Revenue","Revenue_2023" ,"Revenue_2024", "Revenue_2025"),
    ("Rev_Conc","Rev_Conc_2023" ,"Rev_Conc_2024", "Rev_Conc_2025"),
    ("No of Tickets","Tickets_2023","Tickets_2024", "Tickets_2025"),
    ("Average Frequency","Average_Frequency_2023", "Average_Frequency_2024", "Average_Frequency_2025"),
    ("Avg Total Spend per Visit","Avg_Total_Spend_per_Visit_2023" ,"Avg_Total_Spend_per_Visit_2024", "Avg_Total_Spend_per_Visit_2025"),
    ("Avg Ticket Spend per Visit","Avg_Ticket_Spend_per_Visit_2023" ,"Avg_Ticket_Spend_per_Visit_2024", "Avg_Ticket_Spend_per_Visit_2025"),
    ("Avg Concession Spend per visit","Avg_Concession_Spend_per_Visit_2023" ,"Avg_Concession_Spend_per_Visit_2024", "Avg_Concession_Spend_per_Visit_2025"),
    ("Total Spend per person in a year","Total_Spend_per_Person_per_Year_2023", "Total_Spend_per_Person_per_Year_2024", "Total_Spend_per_Person_per_Year_2025"),
    ("Ticket Spend per person in a year","Ticket_Spend_per_Person_per_Year_2023", "Ticket_Spend_per_Person_per_Year_2024", "Ticket_Spend_per_Person_per_Year_2025"),
    ("Concession Spend per person in a year","Concession_Spend_per_Person_per_Year_2023", "Concession_Spend_per_Person_per_Year_2024", "Concession_Spend_per_Person_per_Year_2025"),
    ("Concession Incidence","Concession_Incidence_2023" ,"Concession_Incidence_2024", "Concession_Incidence_2025"),
    ("Total Tickets per Visit","Total_Tickets_per_Visit_2023" ,"Total_Tickets_per_Visit_2024", "Total_Tickets_per_Visit_2025")
]

df_out = reduce(
    lambda a, b: a.unionByName(b),
    [
        df_calc.select(
            "Segment",
            "Category",
            F.lit(metric_label).alias("Metric"),
            F.col(col_2023).alias("Year_2023"),
            F.col(col_2024).alias("Year_2024"),
            F.col(col_2025).alias("Year_2025"),
        )
        for (metric_label, col_2023, col_2024, col_2025) in metric_map
    ]
)

# Segment order: individual -> top3 -> all
seg_order = (
    F.when(F.col("Segment") == "Champions", 1)
     .when(F.col("Segment") == "Potential Loyalist", 2)
     .when(F.col("Segment") == "Value Conscious Loyals", 3)
     .when(F.col("Segment") == "Top 3 Segments", 4)
     .when(F.col("Segment") == "All Segments", 5)
     .otherwise(999)
)

# Category order: No then Yes
cat_order = (
    F.when(F.col("Category") == "No", 1)
     .when(F.col("Category") == "Yes", 2)
     .otherwise(999)
)

# Metric order
order_map = F.create_map(*[
    x for kv in [(m, i) for i, m in enumerate(metric_order_labels, start=1)]
    for x in (F.lit(kv[0]), F.lit(kv[1]))
])

df_out = (
    df_out
    .withColumn("_seg_order", seg_order)
    .withColumn("_cat_order", cat_order)
    .withColumn("_metric_order", order_map[F.col("Metric")])
    .orderBy("_seg_order", "_cat_order", "_metric_order")
    .drop("_seg_order", "_cat_order", "_metric_order")
)


In [0]:
df_out.display()

In [0]:
cte_cc = (
    T_events.alias("p")
    .join(
        T_customer.alias("c"),
        F.col("c.cc_user_profile_id") == F.col("p.cc_user_profile_id"),
        "left",
    )
    .filter(F.col("p.reporting_datetime").isNotNull())
    .filter(F.year(F.col("p.reporting_datetime")) == '2025')
    .filter(F.col("p.plan_id") == F.lit("monthly1"))
    .groupBy(F.col("p.cc_user_profile_id"), F.col("c.CDE_ID"))
    .agg(
        F.count(F.when(F.col("p.event_type") == "subscription_created", F.col("p.reporting_datetime"))).alias("No_of_times_created"),
        F.count(F.when(F.col("p.event_type") == "subscription_renewed", F.col("p.reporting_datetime"))).alias("No_of_times_renewed"),
    )
)

cte_cc1 = (
    cte_cc
    .withColumn(
        "Cineclub_Category",
        F.when(F.col("No_of_times_created") >= 2, F.lit("Samplers"))
         .when((F.col("No_of_times_created") == 1) & (F.col("No_of_times_renewed") == 0), F.lit("One time subscribers"))
         .when((F.col("No_of_times_created") == 1) & (F.col("No_of_times_renewed").between(1, 4)), F.lit("Seasonal goers"))
         .otherwise(F.lit("Program lovers"))
    )
)

cte_2025_rfm_with_cc_category = (
    cte_2025_rfm_member.alias("r")
    .filter(F.col("r.Category") == "Yes")
    .join(cte_cc1.select("CDE_ID", "Cineclub_Category").alias("cc"), on="CDE_ID", how="inner")
)



In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ============================================================
# INPUTS:
#   cte_2025_rfm_member  (CDE_ID-level, has Segment/Category + 2025 metrics)
#   cte_cc1              (CDE_ID -> Cineclub_Category)
# OUTPUT:
#   df_out_cc_cat  (Segment x Metric, pivoted by Cineclub_Category)
# NOTE:
#   - ONLY per-segment results (no Top3, no All)
# ============================================================

def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

# 0) Attach Cineclub_Category (only Category="Yes")
df_cc = (
    cte_2025_rfm_member
    .filter(F.col("Category") == "Yes")
    .join(cte_cc1.select("CDE_ID", "Cineclub_Category"), on="CDE_ID", how="inner")
)

# 1) Segment x Cineclub_Category rollup (summing member-level metrics)
df_seg_cat = (
    df_cc
    .groupBy("Cineclub_Category")
    .agg(
        F.sum(F.col("No_of_Members").cast("double")).alias("No_of_members"),
        F.sum(F.col("No_of_Visits_2025").cast("double")).alias("No_of_Visits"),
        F.sum(F.col("Visits_with_Concession_2025").cast("double")).alias("Visits_with_Concession"),
        F.sum(F.col("Net_Revenue_2025").cast("double")).alias("Net_Revenue"),
        F.sum(F.col("Net_Revenue_Tickets_2025").cast("double")).alias("Net_Revenue_Tickets"),
        F.sum(F.col("Net_Revenue_Concession_2025").cast("double")).alias("Net_Revenue_Concession"),
        F.sum(F.col("No_of_Regular_Tickets_2025").cast("double")).alias("No_of_Regular_Tickets"),
        F.sum(F.col("No_of_Premium_Tickets_2025").cast("double")).alias("No_of_Premium_Tickets"),
        F.sum(F.col("Member_Tickets").cast("double")).alias("Member_Tickets"),
        F.sum(F.col("Companion_Tickets").cast("double")).alias("Companion_Tickets"),
    )
)

# 2) Derived metrics
df_calc = (
    df_seg_cat
    .withColumn("Avg Frequency", sdiv(F.col("No_of_Visits"), F.col("No_of_members")))
    .withColumn("Avg Total Spend per visit", sdiv(F.col("Net_Revenue"), F.col("No_of_Visits")))
    .withColumn("Avg Ticket Spend per visit", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Avg Concession Spend per visit", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Spend per person in a year", sdiv(F.col("Net_Revenue"), F.col("No_of_members")))
    .withColumn("Ticket Spend per person in a year", sdiv(F.col("Net_Revenue") - F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Spend per person in a year", sdiv(F.col("Net_Revenue_Concession"), F.col("No_of_members")))
    .withColumn("Concession Incidence", sdiv(F.col("Visits_with_Concession"), F.col("No_of_Visits")))
    .withColumn("Total Tickets per Visit", sdiv(F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets"), F.col("No_of_Visits")))
    .withColumn("999 Tickets per Visit", sdiv(F.col("Member_Tickets") + F.col("Companion_Tickets"), F.col("No_of_Visits")))
    .withColumn(
        "Uptake Premium Experiences",
        sdiv(F.col("No_of_Premium_Tickets"), (F.col("No_of_Regular_Tickets") + F.col("No_of_Premium_Tickets")))
    )
    .withColumn(
        "Uptake Companion Experiences",
        sdiv(F.col("Companion_Tickets"), F.col("Member_Tickets"))
    )
)

# 3) Metrics as rows, pivot by Cineclub_Category
metric_order = [
    "No_of_members",
    "No_of_Visits",
    "Visits_with_Concession",
    "Net_Revenue",
    "Net_Revenue_Concession",
    "No_of_Regular_Tickets",
    "No_of_Premium_Tickets",
    "Member_Tickets",
    "Companion_Tickets",
    "Avg Frequency",
    "Avg Total Spend per visit",
    "Avg Ticket Spend per visit",
    "Avg Concession Spend per visit",
    "Total Spend per person in a year",
    "Ticket Spend per person in a year",
    "Concession Spend per person in a year",
    "Concession Incidence",
    "Total Tickets per Visit",
    "999 Tickets per Visit",
    "Uptake Premium Experiences",
    "Uptake Companion Experiences"
]

df_long = reduce(
    lambda a, b: a.unionByName(b),
    [df_calc.select("Cineclub_Category", F.lit(m).alias("Metric"), F.col(m).alias("Value"))
     for m in metric_order]
)

df_out_cc_cat = (
    df_long
    .groupBy("Metric")
    .pivot("Cineclub_Category")
    .agg(F.first("Value"))
)

# 4) Enforce metric order within each segment
order_map = F.create_map(*[
    x for kv in [(m, i) for i, m in enumerate(metric_order, start=1)]
    for x in (F.lit(kv[0]), F.lit(kv[1]))
])

df_out_cc_cat = (
    df_out_cc_cat
    .withColumn("_metric_order", order_map[F.col("Metric")])
    .orderBy("_metric_order")
    .drop("_metric_order")
)




In [0]:
display(df_out_cc_cat)

In [0]:
final_cc_rollup = (
    cte_cc1.alias("cc")
    .join(cte_2024.alias("y19"), F.col("y19.ID_2024") == F.col("cc.CDE_ID"), "inner")
    .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("cc.CDE_ID"), "inner")
    .groupBy("cc.Cineclub_Category")
    .agg(
        F.count("cc.CDE_ID").alias("No_of_members"),

        F.sum("y25.No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("y25.Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("y25.Revenue_2025").alias("Revenue_2025"),
        F.sum("y25.Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("y25.Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("y25.Tickets_2025").alias("Tickets_2025"),

        F.sum("y19.No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("y19.Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("y19.Revenue_2024").alias("Revenue_2024"),
        F.sum("y19.Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("y19.Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("y19.Tickets_2024").alias("Tickets_2024"),
    )
)


In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ----------------------------
# safe divide
# ----------------------------
def sdiv(num, den):
    return F.when((den.isNull()) | (den == 0), F.lit(None)).otherwise(num / den)

# ============================================================
# 1) Your existing rollup (as-is)
# ============================================================
final_cc_rollup_base = (
    cte_cc1.alias("cc")
    .join(cte_2024.alias("y19"), F.col("y19.ID_2024") == F.col("cc.CDE_ID"), "inner")
    .join(cte_2025.alias("y25"), F.col("y25.ID_2025") == F.col("cc.CDE_ID"), "inner")
    .groupBy("cc.Cineclub_Category")
    .agg(
        F.count("cc.CDE_ID").alias("No_of_members"),

        F.sum("y25.No_of_Visits_2025").alias("No_of_Visits_2025"),
        F.sum("y25.Visits_with_Concession_2025").alias("Visits_with_Concession_2025"),
        F.sum("y25.Revenue_2025").alias("Revenue_2025"),
        F.sum("y25.Rev_Conc_2025").alias("Rev_Conc_2025"),
        F.sum("y25.Rev_Tickets_2025").alias("Rev_Tickets_2025"),
        F.sum("y25.Tickets_2025").alias("Tickets_2025"),

        F.sum("y19.No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("y19.Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("y19.Revenue_2024").alias("Revenue_2024"),
        F.sum("y19.Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("y19.Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("y19.Tickets_2024").alias("Tickets_2024"),
    )
)

# ============================================================
# 2) Derived metrics (same style as your model, but for both years)
#    NOTE: since you only have total Tickets (not Regular/Premium/999),
#          the ticket-format metrics are not computed here.
# ============================================================
final_cc_rollup_calc = (
    final_cc_rollup_base
    # ---- 2025 ----
    .withColumn("Avg Frequency_2025", sdiv(F.col("No_of_Visits_2025"), F.col("No_of_members")))
    .withColumn("Avg Total Spend per visit_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg Ticket Spend per visit_2025", sdiv(F.col("Rev_Tickets_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Avg Concession Spend per visit_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Total Spend per person in a year_2025", sdiv(F.col("Revenue_2025"), F.col("No_of_members")))
    .withColumn("Ticket Spend per person in a year_2025", sdiv(F.col("Rev_Tickets_2025"), F.col("No_of_members")))
    .withColumn("Concession Spend per person in a year_2025", sdiv(F.col("Rev_Conc_2025"), F.col("No_of_members")))
    .withColumn("Concession Incidence_2025", sdiv(F.col("Visits_with_Concession_2025"), F.col("No_of_Visits_2025")))
    .withColumn("Tickets per Visit_2025", sdiv(F.col("Tickets_2025"), F.col("No_of_Visits_2025")))

    # ---- 2024 ----
    .withColumn("Avg Frequency_2024", sdiv(F.col("No_of_Visits_2024"), F.col("No_of_members")))
    .withColumn("Avg Total Spend per visit_2024", sdiv(F.col("Revenue_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Avg Ticket Spend per visit_2024", sdiv(F.col("Rev_Tickets_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Avg Concession Spend per visit_2024", sdiv(F.col("Rev_Conc_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Total Spend per person in a year_2024", sdiv(F.col("Revenue_2024"), F.col("No_of_members")))
    .withColumn("Ticket Spend per person in a year_2024", sdiv(F.col("Rev_Tickets_2024"), F.col("No_of_members")))
    .withColumn("Concession Spend per person in a year_2024", sdiv(F.col("Rev_Conc_2024"), F.col("No_of_members")))
    .withColumn("Concession Incidence_2024", sdiv(F.col("Visits_with_Concession_2024"), F.col("No_of_Visits_2024")))
    .withColumn("Tickets per Visit_2024", sdiv(F.col("Tickets_2024"), F.col("No_of_Visits_2024")))
)

# ============================================================
# 3) Reshape: metrics as rows, and show both years as columns
#    Output: Cineclub_Category | Metric | Year_2024 | Year_2025
# ============================================================
metric_order_labels = [
    "No. of Members",
    "No. of Visits",
    "Visits with Concession",
    "Revenue",
    "Rev_Conc",
    "Rev_Tickets",
    "Tickets",
    "Avg Frequency",
    "Avg Total Spend per visit",
    "Avg Ticket Spend per visit",
    "Avg Concession Spend per visit",
    "Total Spend per person in a year",
    "Ticket Spend per person in a year",
    "Concession Spend per person in a year",
    "Concession Incidence",
    "Tickets per Visit",
]

# display label -> (2024 col, 2025 col)
metric_map = {
    "No. of Members": ("No_of_members", "No_of_members"),
    "No. of Visits": ("No_of_Visits_2024", "No_of_Visits_2025"),
    "Visits with Concession": ("Visits_with_Concession_2024", "Visits_with_Concession_2025"),
    "Revenue": ("Revenue_2024", "Revenue_2025"),
    "Rev_Conc": ("Rev_Conc_2024", "Rev_Conc_2025"),
    "Rev_Tickets": ("Rev_Tickets_2024", "Rev_Tickets_2025"),
    "Tickets": ("Tickets_2024", "Tickets_2025"),
    "Avg Frequency": ("Avg Frequency_2024", "Avg Frequency_2025"),
    "Avg Total Spend per visit": ("Avg Total Spend per visit_2024", "Avg Total Spend per visit_2025"),
    "Avg Ticket Spend per visit": ("Avg Ticket Spend per visit_2024", "Avg Ticket Spend per visit_2025"),
    "Avg Concession Spend per visit": ("Avg Concession Spend per visit_2024", "Avg Concession Spend per visit_2025"),
    "Total Spend per person in a year": ("Total Spend per person in a year_2024", "Total Spend per person in a year_2025"),
    "Ticket Spend per person in a year": ("Ticket Spend per person in a year_2024", "Ticket Spend per person in a year_2025"),
    "Concession Spend per person in a year": ("Concession Spend per person in a year_2024", "Concession Spend per person in a year_2025"),
    "Concession Incidence": ("Concession Incidence_2024", "Concession Incidence_2025"),
    "Tickets per Visit": ("Tickets per Visit_2024", "Tickets per Visit_2025"),
}

df_long = reduce(
    lambda a, b: a.unionByName(b),
    [
        final_cc_rollup_calc.select(
            F.col("Cineclub_Category"),
            F.lit(m).alias("Metric"),
            F.col(metric_map[m][0]).alias("Year_2024"),
            F.col(metric_map[m][1]).alias("Year_2025"),
        )
        for m in metric_order_labels
    ]
)

# enforce row order
order_map = F.create_map(*[
    x for kv in [(m, i) for i, m in enumerate(metric_order_labels, start=1)]
    for x in (F.lit(kv[0]), F.lit(kv[1]))
])

final_cc_rollup_out = (
    df_long
    .withColumn("_metric_order", order_map[F.col("Metric")])
    .orderBy("Cineclub_Category", "_metric_order")
    .drop("_metric_order")
)




In [0]:
display(final_cc_rollup_out)